# Classificador WiSARD com TF-IDF + termômetro 

## 1. Configuração

In [1]:
import os
import time
from collections import Counter

import numpy as np
import scipy.sparse as sp
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, classification_report, confusion_matrix)
import wisardpkg as wsd  # Importa a biblioteca WiSARD

# Pasta onde os arquivos CSV estão localizados (saida da opcao 3 do __init__.py)
pasta_dados = 'csv/aTribuna-21dir/'

# Arquivo com os termos e seus IDFs (saida da opcao 2), usado so para mostrar nomes dos termos
arquivo_fi = 'arq_fi/fi.txt'

# Quantidade de termos mantidos pela selecao chi2
K_TERMOS = 7000

# Quantidade de bits do termometro para cada termo
BITS_TERMOMETRO = 8

# Bits de enderecamento de cada RAM da WiSARD
ENDERECAMENTO = 5

## 2. Leitura dos vetores ponderados

Cada linha de `csv/<classe>.csv` é um documento: a primeira coluna é a classe e as demais são os pesos TF-IDF,
calculados como `idf × (1 + ln(tf))`. Peso `0` significa que o termo não aparece no documento.

Como quase todos os pesos são zero, a matriz é guardada em formato **esparso** (só os valores
diferentes de zero). Ela só volta a ser densa depois da seleção de termos.

In [2]:
def ler_dados_arquivo(arquivo_csv):
    # Lê o CSV (gerado pela opcao 3, sem cabecalho) linha a linha guardando so os
    # pesos diferentes de zero (matriz esparsa): quase todo peso e zero, e a matriz
    # densa com dezenas de milhares de termos nao cabe na memoria.
    rotulos = []
    pesos = []
    colunas = []
    inicio_linhas = [0]
    qtd_termos = 0
    with open(arquivo_csv, 'r') as f:
        for linha in f:
            linha = linha.strip()
            if not linha:
                continue
            rotulo, valores = linha.split(',', 1)
            vetor = np.array(valores.split(','), dtype=np.float32)
            qtd_termos = len(vetor)
            presentes = np.flatnonzero(vetor)
            rotulos.append(rotulo)
            pesos.append(vetor[presentes])
            colunas.append(presentes)
            inicio_linhas.append(inicio_linhas[-1] + len(presentes))

    # rótulos (primeira coluna) e pesos TF-IDF (demais colunas)
    X = sp.csr_matrix((np.concatenate(pesos), np.concatenate(colunas), np.array(inicio_linhas)),
                      shape=(len(rotulos), qtd_termos))
    return X, np.array(rotulos)


todos_X = []
todos_y = []

for arquivo in sorted(os.listdir(pasta_dados)):
    if arquivo.endswith(".csv"):
        X, y = ler_dados_arquivo(os.path.join(pasta_dados, arquivo))
        print(f"{arquivo:<15} {X.shape[0]:>6} documentos  {X.shape[1]} termos")
        todos_X.append(X)
        todos_y.append(y)

# Concatena todos os dados em uma matriz única
X = sp.vstack(todos_X).tocsr()
y = np.concatenate(todos_y)
del todos_X, todos_y  # libera as copias por classe

print(f"\nTotal: {X.shape[0]} documentos x {X.shape[1]} termos, {len(set(y))} classes")
memoria = X.data.nbytes + X.indices.nbytes + X.indptr.nbytes
print(f"Memória da matriz TF-IDF (esparsa): {memoria / 1e6:.1f} MB "
      f"(densa seria {X.shape[0] * X.shape[1] * 4 / 1e9:.2f} GB)")

at2.csv           5618 documentos  27444 termos
eco.csv           6557 documentos  27444 termos
esp.csv           1470 documentos  27444 termos

Total: 13645 documentos x 27444 termos, 3 classes
Memória da matriz TF-IDF (esparsa): 21.0 MB (densa seria 1.50 GB)


### Como é um vetor ponderado

In [3]:
# nomes dos termos na mesma ordem das colunas do CSV
termos = []
if os.path.exists(arquivo_fi):
    with open(arquivo_fi, 'r', encoding='iso-8859-1') as f:
        for linha in f:
            partes = linha.split(":")
            if len(partes) == 2:
                termos.append(partes[0].strip())


def nome_termo(i):
    return termos[i] if i < len(termos) else f"termo {i}"


doc = 0
vetor = X[doc].toarray().ravel()
presentes = np.nonzero(vetor)[0]
print(f"Documento {doc} (classe {y[doc]}): {len(presentes)} de {len(vetor)} termos com peso > 0\n")

print("10 termos de maior peso:")
for i in presentes[np.argsort(vetor[presentes])[::-1][:10]]:
    print(f"  {nome_termo(i):<20} {vetor[i]:.2f}")

esparsidade = 1 - X.nnz / (X.shape[0] * X.shape[1])
print(f"\nEsparsidade da matriz: {esparsidade * 100:.2f}% dos pesos são zero")

Documento 0 (classe aTribuna-21dir/at2): 77 de 27444 termos com peso > 0

10 termos de maior peso:
  chat                 7.91
  Tony                 7.81
  dedinho              7.73
  gordinha             7.58
  testosteronar        7.04
  euforico             6.96
  esmalte              6.75
  metafora             6.75
  macho                6.69
  fofo                 6.48

Esparsidade da matriz: 99.30% dos pesos são zero


## 3. Divisão treino / teste (80% / 20%)

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)
del X  # treino e teste ja sao copias

print(f"Treino: {X_train.shape[0]} documentos")
print(f"Teste:  {X_test.shape[0]} documentos\n")
print("Documentos de treino por classe:")
for classe, qtd in sorted(Counter(y_train).items()):
    print(f"  {classe:<10} {qtd}")

Treino: 10916 documentos
Teste:  2729 documentos

Documentos de treino por classe:
  aTribuna-21dir/at2 4478
  aTribuna-21dir/eco 5275
  aTribuna-21dir/esp 1163


## 4. Seleção de termos (chi²)

Mesmo após o corte da opção 2, o vocabulário tem dezenas de milhares de termos. O **chi²** mede o quanto cada
termo está associado às classes: termos que aparecem muito mais em uma classe do que nas outras têm valor alto.
Ficamos com os `K_TERMOS` de maior chi².

A seleção é calculada **somente no treino** e depois aplicada ao teste, para que o teste não influencie quais
termos são escolhidos.

In [5]:
qtd_termos = X_train.shape[1]
seletor = SelectKBest(chi2, k=min(K_TERMOS, qtd_termos)).fit(X_train, y_train)
selecionados = seletor.get_support(indices=True)

# depois da selecao a matriz e pequena e pode voltar a ser densa (o termometro precisa)
X_train = seletor.transform(X_train).toarray()
X_test = seletor.transform(X_test).toarray()
print(f"Termos: {qtd_termos} -> {X_train.shape[1]}\n")

print("20 termos com maior chi²:")
for i in selecionados[np.argsort(seletor.scores_[selecionados])[::-1][:20]]:
    print(f"  {nome_termo(i):<20} {seletor.scores_[i]:>10.1f}")

# daqui em diante as colunas sao so os termos selecionados
termos = [nome_termo(i) for i in selecionados]

# documentos que ficaram sem nenhum termo selecionado (vetor todo zero)
vazios_treino = int((X_train.sum(axis=1) == 0).sum())
vazios_teste = int((X_test.sum(axis=1) == 0).sum())
print(f"\nDocumentos sem nenhum termo selecionado: treino {vazios_treino}, teste {vazios_teste}")

Termos: 27444 -> 7000

20 termos com maior chi²:
  policer                  6765.3
  proscholdtr              6759.2
  criminoso                6693.2
  bandido                  6391.5
  atriz                    6277.3
  filme                    5811.3
  ator                     5655.2
  crime                    5321.0
  novelo                   5306.5
  policial                 5291.2
  onibu                    5179.6
  cantor                   4920.4
  prender                  4903.2
  assalto                  4855.3
  nnn                      4811.3
  delegar                  4551.3
  personagem               4530.3
  show                     4443.5
  aliner                   4439.8
  droga                    4154.5

Documentos sem nenhum termo selecionado: treino 7, teste 1


## 5. Termômetro

A WiSARD só aceita entradas **binárias**, mas o TF-IDF é um número real. O termômetro converte cada peso em
`BITS_TERMOMETRO` bits:

1. normaliza o peso do termo para `[0, 1]` usando o mínimo e o máximo **daquele termo no treino**
   (o teste não é usado para não vazar informação);
2. liga o bit `j` se o valor normalizado for maior que `j / bits`.

Quanto maior o peso, mais bits ficam ligados, como a coluna de um termômetro. Com 8 bits:

| peso normalizado | bits |
|---|---|
| 0 (termo ausente) | `00000000` |
| 0.1 | `10000000` |
| 0.5 | `11110000` |
| 1.0 | `11111111` |

Um vetor de `T` termos vira um vetor binário de `T × BITS_TERMOMETRO` posições.

In [6]:
class Termometro:

    def __init__(self, bits):
        self.bits = bits
        self.minimo = None
        self.maximo = None

    def ajustar(self, X):
        # min e max de cada termo calculados somente no treino
        self.minimo = X.min(axis=0)
        self.maximo = X.max(axis=0)
        return self

    def normalizar(self, X):
        faixa = self.maximo - self.minimo
        faixa[faixa == 0] = 1  # evita divisao por zero em termos constantes
        return np.clip((X - self.minimo) / faixa, 0, 1)

    def transformar(self, X):
        # bit j ligado se o valor normalizado passa de j/bits
        limiares = np.arange(self.bits) / self.bits
        X_bin = self.normalizar(X)[:, :, None] > limiares

        # (docs, termos, bits) -> (docs, termos*bits)
        return X_bin.reshape(X.shape[0], -1).astype(np.uint8)


# exemplo pequeno: 3 documentos x 2 termos, termometro de 4 bits
exemplo = np.array([[0.0, 1.0],
                    [1.5, 2.0],
                    [3.0, 4.0]])
t = Termometro(4).ajustar(exemplo)
print("pesos:\n", exemplo)
print("\nnormalizado:\n", t.normalizar(exemplo))
print("\nbinario (4 bits do termo 1 | 4 bits do termo 2):")
for linha in t.transformar(exemplo):
    print(" ", "".join(map(str, linha[:4])), "|", "".join(map(str, linha[4:])))

pesos:
 [[0.  1. ]
 [1.5 2. ]
 [3.  4. ]]

normalizado:
 [[0.         0.        ]
 [0.5        0.33333333]
 [1.         1.        ]]

binario (4 bits do termo 1 | 4 bits do termo 2):
  0000 | 0000
  1100 | 1100
  1111 | 1111


### Aplicando nos dados reais

In [7]:
termometro = Termometro(BITS_TERMOMETRO).ajustar(X_train)
X_train_bin = termometro.transformar(X_train)
X_test_bin = termometro.transformar(X_test)

print(f"Treino: {X_train.shape} -> {X_train_bin.shape}")
print(f"Teste:  {X_test.shape} -> {X_test_bin.shape}")
print(f"Memória da entrada binária: {(X_train_bin.nbytes + X_test_bin.nbytes) / 1e9:.2f} GB (uint8)")
print(f"Bits ligados no treino: {X_train_bin.mean() * 100:.2f}%\n")

# mostra o que aconteceu com os termos do primeiro documento de treino
vetor = X_train[0]
norm = termometro.normalizar(X_train[:1])[0]
bits = X_train_bin[0].reshape(-1, BITS_TERMOMETRO)
print(f"Documento de treino 0 (classe {y_train[0]}):")
print(f"  {'termo':<20} {'peso':>6} {'normal.':>8}  bits")
for i in np.argsort(vetor)[::-1][:8]:
    print(f"  {nome_termo(i):<20} {vetor[i]:>6.2f} {norm[i]:>8.2f}  {''.join(map(str, bits[i]))}")

Treino: (10916, 7000) -> (10916, 56000)
Teste:  (2729, 7000) -> (2729, 56000)
Memória da entrada binária: 0.76 GB (uint8)
Bits ligados no treino: 0.86%

Documento de treino 0 (classe aTribuna-21dir/eco):
  termo                  peso  normal.  bits
  quadrilha             11.75     0.85  11111110
  inss                  11.57     0.81  11111110
  fraude                11.17     0.72  11111100
  previdenciario        10.70     0.71  11111100
  retroativo             9.57     0.51  11111000
  denominar              8.75     1.00  11111111
  idoso                  8.60     0.56  11111000
  auditoria              8.50     0.55  11111000


## 6. Treinamento da WiSARD

A WiSARD tem um **discriminador por classe**. Cada discriminador divide a entrada binária em tuplas de
`ENDERECAMENTO` bits (em posições embaralhadas) e cada tupla endereça uma **RAM**.

- **Treino:** para cada documento, as RAMs do discriminador da sua classe marcam os endereços vistos.
- **Classificação:** cada discriminador conta quantas RAMs reconhecem o endereço do documento; vence a classe com
  a maior contagem.

Quantidade de RAMs por discriminador = `tamanho da entrada / ENDERECAMENTO`.

In [8]:
model = wsd.Wisard(ENDERECAMENTO)

tam_entrada = X_train_bin.shape[1]
print(f"Entrada: {tam_entrada} bits -> {int(np.ceil(tam_entrada / ENDERECAMENTO))} RAMs por discriminador")
print(f"Discriminadores (classes): {len(set(y_train))}\n")

inicio = time.time()
model.train(X_train_bin.tolist(), y_train.tolist())
print(f"Treino concluído em {time.time() - inicio:.1f}s")

Entrada: 56000 bits -> 11200 RAMs por discriminador
Discriminadores (classes): 3

Treino concluído em 69.2s


## 7. Classificação do conjunto de teste

In [9]:
inicio = time.time()
y_pred = np.array(model.classify(X_test_bin.tolist()))
print(f"Classificação concluída em {time.time() - inicio:.1f}s\n")

print("Primeiras 10 previsões:")
for real, prev in zip(y_test[:10], y_pred[:10]):
    print(f"  real: {real:<10} previsto: {prev:<10} {'OK' if real == prev else 'ERRO'}")

Classificação concluída em 16.9s

Primeiras 10 previsões:
  real: aTribuna-21dir/eco previsto: aTribuna-21dir/eco OK
  real: aTribuna-21dir/esp previsto: aTribuna-21dir/eco ERRO
  real: aTribuna-21dir/eco previsto: aTribuna-21dir/eco OK
  real: aTribuna-21dir/at2 previsto: aTribuna-21dir/at2 OK
  real: aTribuna-21dir/at2 previsto: aTribuna-21dir/at2 OK
  real: aTribuna-21dir/at2 previsto: aTribuna-21dir/at2 OK
  real: aTribuna-21dir/at2 previsto: aTribuna-21dir/at2 OK
  real: aTribuna-21dir/at2 previsto: aTribuna-21dir/at2 OK
  real: aTribuna-21dir/at2 previsto: aTribuna-21dir/at2 OK
  real: aTribuna-21dir/eco previsto: aTribuna-21dir/eco OK


## 8. Métricas

In [10]:
acuracia = accuracy_score(y_test, y_pred)
precision_micro = precision_score(y_test, y_pred, average='micro', zero_division=1)
precision_macro = precision_score(y_test, y_pred, average='macro', zero_division=1)
recall_micro = recall_score(y_test, y_pred, average='micro', zero_division=1)
recall_macro = recall_score(y_test, y_pred, average='macro', zero_division=1)
f1_micro = f1_score(y_test, y_pred, average='micro', zero_division=1)
f1_macro = f1_score(y_test, y_pred, average='macro', zero_division=1)

print(f'Acurácia do modelo WiSARD: {acuracia * 100:.2f}%')
print(f'Precisão do modelo WiSARD (Micro): {precision_micro * 100:.2f}%')
print(f'Precisão do modelo WiSARD (Macro): {precision_macro * 100:.2f}%')
print(f'Sensibilidade (Recall) do modelo WiSARD (Micro): {recall_micro * 100:.2f}%')
print(f'Sensibilidade (Recall) do modelo WiSARD (Macro): {recall_macro * 100:.2f}%')
print(f'Pontuação F1 do modelo WiSARD (Micro): {f1_micro * 100:.2f}%')
print(f'Pontuação F1 do modelo WiSARD (Macro): {f1_macro * 100:.2f}%')

Acurácia do modelo WiSARD: 88.53%
Precisão do modelo WiSARD (Micro): 88.53%
Precisão do modelo WiSARD (Macro): 82.27%
Sensibilidade (Recall) do modelo WiSARD (Micro): 88.53%
Sensibilidade (Recall) do modelo WiSARD (Macro): 84.47%
Pontuação F1 do modelo WiSARD (Micro): 88.53%
Pontuação F1 do modelo WiSARD (Macro): 83.17%


### Métricas por classe

In [11]:
print(classification_report(y_test, y_pred, zero_division=0))

                    precision    recall  f1-score   support

aTribuna-21dir/at2       0.92      0.96      0.94      1140
aTribuna-21dir/eco       0.93      0.86      0.89      1282
aTribuna-21dir/esp       0.62      0.71      0.66       307

          accuracy                           0.89      2729
         macro avg       0.82      0.84      0.83      2729
      weighted avg       0.89      0.89      0.89      2729



### Matriz de confusão (linhas = classe real, colunas = classe prevista)

In [12]:
classes = sorted(set(y_test) | set(y_pred))
matriz = confusion_matrix(y_test, y_pred, labels=classes)

larg = max(len(c) for c in classes) + 2
print(" " * larg + "".join(f"{c:>{larg}}" for c in classes))
for classe, linha in zip(classes, matriz):
    print(f"{classe:<{larg}}" + "".join(f"{v:>{larg}}" for v in linha))

                      aTribuna-21dir/at2  aTribuna-21dir/eco  aTribuna-21dir/esp
aTribuna-21dir/at2                  1100                  25                  15
aTribuna-21dir/eco                    64                1097                 121
aTribuna-21dir/esp                    36                  52                 219


## 9. Ajustes possíveis

- **`K_TERMOS`**: menos termos = entrada menor e mais rápida, mas pode perder informação. Teste 1000, 3000 e 5000.
- **`MIN_DOCS` / `MAX_PROP_DOCS`** (no `__init__.py`): controlam o corte do vocabulário na opção 2; mudar exige
  rodar as opções 2 e 3 de novo.
- **`BITS_TERMOMETRO`**: mais bits = mais resolução do peso, mas entrada maior (mais memória e tempo).
  Teste 4, 8 e 16.
- **`ENDERECAMENTO`**: tuplas maiores deixam a rede mais específica (memoriza mais, generaliza menos);
  menores generalizam mais e podem saturar as RAMs. Valores comuns ficam entre 4 e 32.